In [ ]:
_NB_NAME_ = "Linear Regression"
# @title 🤖 AI Feedback — enter your username and click ▶ Run
# @markdown **No API key needed.** AI grading runs free inside your Colab session.
# @markdown
# @markdown Enter your GitHub username, then click ▶ Run for question-by-question feedback.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"e.g. jsmith42"}

# ── runs automatically — do not edit below ───────────────────
import json, textwrap, re as _re, time
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_quiz_questions():
    """Pull question text from the quiz cell @markdown lines."""
    try:
        import ipynbname
        nb_path = ipynbname.path()
        with open(nb_path) as f:
            nb = json.load(f)
        for cell in nb["cells"]:
            src = "".join(cell.get("source", []))
            if "@title" in src and "Quiz" in src:
                qs = re.findall(r"@markdown \*\*Q\d+:\*\* (.+)", src)
                if qs: return qs
    except Exception:
        pass
    return []

def _call_gemini(prompt, max_retries=3):
    """Call Gemini with retry on 429 rate limit."""
    last_err = None
    for attempt in range(max_retries):
        try:
            import google.generativeai as genai
            import google.auth, google.auth.transport.requests
            creds, _ = google.auth.default()
            creds.refresh(google.auth.transport.requests.Request())
            genai.configure(credentials=creds)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(
                prompt,
                generation_config={"max_output_tokens": 1500, "temperature": 0.3}
            )
            return result.text, "Gemini via Colab"
        except Exception as e:
            last_err = str(e)
            if "429" in str(e) or "quota" in str(e).lower():
                wait = 2 ** attempt
                print(f"  Rate limit hit — waiting {wait}s before retry {attempt+1}/{max_retries}...")
                time.sleep(wait)
                continue
            break
    # Try stored key as fallback
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=key)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(prompt)
            return result.text, "Gemini via key"
    except Exception:
        pass
    return None, last_err

def _build_prompt(answers_dict, nb_name, questions):
    answered  = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total   = len(answers_dict)
    n_done    = len(answered)

    # Pair each question with the student answer
    qa_pairs = ""
    for i, (k, v) in enumerate(answers_dict.items(), 1):
        q_text = questions[i-1] if i-1 < len(questions) else f"Question {i}"
        a_text = str(v).strip() if str(v).strip() else "(no answer)"
        qa_pairs += f"Q{i}: {q_text}\nA{i}: {a_text}\n\n"

    return f"""You are a TA grading a student quiz for the "{nb_name}" notebook in a data science course called "Data Pattern Recognition for the Rest of Us" (based on ISLP and business analytics).

{qa_pairs.strip()}

For EACH question:
- Decide if the answer is CORRECT, PARTIALLY CORRECT, or INCORRECT
- A short paraphrase or reasonable approximation counts as CORRECT — do not require exact wording
- For INCORRECT or PARTIAL: name the specific concept they need to review (1 sentence)
- For CORRECT: brief confirmation of what they got right (1 sentence)

Then give an overall summary.

Respond ONLY in this exact JSON format (no markdown fences, no extra text):
{{
  "questions": [
    {{
      "q": 1,
      "status": "<CORRECT|PARTIAL|INCORRECT>",
      "comment": "<one specific sentence>"
    }}
  ],
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent|Good|Needs Review|Incomplete>",
  "summary": "<2 sentences overall: strengths and what to revisit>",
  "review_tip": "<one specific concept, chapter, or notebook to review if they struggled — or 'Great work!' if excellent>"
}}

Scoring guide: CORRECT=1pt, PARTIAL=0.5pt (round to nearest int), INCORRECT=0pt."""

def _show(result, username, nb_name, source, questions):
    STATUS_ICON  = {"CORRECT": "\u2705", "PARTIAL": "\u26a0\ufe0f", "INCORRECT": "\u274c"}
    STATUS_COLOR = {"CORRECT": "\033[92m", "PARTIAL": "\033[93m", "INCORRECT": "\033[91m"}
    R = "\033[0m"
    grade = result.get("grade", "?")
    GRADE_COLOR = {"Excellent":"\033[92m","Good":"\033[94m",
                   "Needs Review":"\033[93m","Incomplete":"\033[91m"}
    GC = GRADE_COLOR.get(grade, "")
    n  = len(answers)
    qs = result.get("quiz_score", 0)
    bar = "\u2588"*qs + "\u2591"*(n-qs)

    print("\n" + "\u2550"*60)
    print(f"  \U0001f916 AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by   {source}")
    print("\u2550"*60)
    print(f"  Student  : {'@'+username if username else '\u26a0 set GITHUB_USERNAME above'}")
    print(f"  Grade    : {GC}{grade}{R}")
    print(f"  Score    : {qs}/{n}  [{bar}]")
    print()

    # Question-by-question breakdown
    q_results = result.get("questions", [])
    if q_results:
        print("  \u2500"*29)
        for qr in q_results:
            idx    = qr.get("q", 0) - 1
            status = qr.get("status", "?")
            icon   = STATUS_ICON.get(status, "\u2753")
            color  = STATUS_COLOR.get(status, "")
            comment= qr.get("comment", "")
            q_text = questions[idx] if idx < len(questions) else f"Question {qr.get('q','?')}"
            # Truncate long question text for display
            q_short = q_text[:55] + "..." if len(q_text) > 55 else q_text
            print(f"  {icon} {color}Q{qr.get('q','?')} {status}{R}")
            print(f"     {q_short}")
            if comment:
                for line in textwrap.wrap(comment, 54):
                    print(f"     \u2192 {line}")
            print()

    # Summary
    summary = result.get("summary", "")
    if summary:
        print("  \u2500"*29)
        print("  Overall:")
        for line in textwrap.wrap(summary, 56):
            print(f"  {line}")

    # Review tip
    tip = result.get("review_tip", "")
    if tip and tip != "Great work!":
        print()
        for line in textwrap.wrap(f"\U0001f4d6 Review: {tip}", 56):
            print(f"  {line}")
    elif tip == "Great work!":
        print()
        print("  \U0001f4d6 Great work! Keep going.")

    print("\u2550"*60 + "\n")

def _fallback_grade(answers_dict):
    """Smart fallback — grade on completion only, no length penalty."""
    n  = len(answers_dict)
    nd = sum(1 for v in answers_dict.values() if str(v).strip())
    if nd == 0:
        return {"quiz_score":0,"grade":"Incomplete",
                "questions":[],
                "summary":"No answers provided — fill in the quiz above.",
                "review_tip":"Complete the quiz and re-run for AI feedback."}
    elif nd == n:
        return {"quiz_score":n,"grade":"Good",
                "questions":[],
                "summary":f"All {n} questions answered. AI grading was unavailable — re-run to get question-by-question feedback.",
                "review_tip":"Re-run this cell in a few minutes for detailed AI feedback."}
    else:
        return {"quiz_score":nd,"grade":"Needs Review",
                "questions":[],
                "summary":f"{nd}/{n} questions answered. Complete the remaining {n-nd} and re-run.",
                "review_tip":"Answer all questions for full feedback."}

# ── Main ──────────────────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    nd       = sum(1 for v in answers.values() if str(v).strip())
    username = GITHUB_USERNAME.strip()
    questions = _get_quiz_questions()

    print(f"\n  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    if username: print(f"  Student  : @{username}")
    print(f"  Grading  : please wait...\n")

    prompt     = _build_prompt(answers, _NB_TITLE, questions)
    raw, src   = _call_gemini(prompt)

    if raw:
        try:
            clean  = _re.sub(r"```(?:json)?\s*|```","",raw).strip()
            result = json.loads(clean)
        except Exception:
            nd2    = sum(1 for v in answers.values() if str(v).strip())
            result = {"quiz_score":nd2,"grade":"Good" if nd2==len(answers) else "Needs Review",
                      "questions":[],"summary":raw[:400],"review_tip":""}
    else:
        if src: print(f"  \u26a0 Gemini unavailable ({src[:60]}) \u2014 showing completion feedback\n")
        result = _fallback_grade(answers)

    _show(result, username, _NB_TITLE, src if raw else None, questions)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")


# 📈 Linear Regression
**ISLP Chapter 3 · Pattern Recognition for the Rest of Us**

> The foundation of supervised learning. Linear regression is simple enough to understand completely, powerful enough to use in production, and teaches concepts (coefficients, p-values, R², residuals) that appear in every model you'll ever build.

---
### What you'll learn
- Simple and multiple linear regression from scratch
- How to interpret every number in the regression output (β, SE, t-stat, p-value, R², RSE)
- Checking assumptions: linearity, homoscedasticity, normality of residuals
- Categorical predictors (dummy variables)
- When linear regression fails and what to do

### Dataset
**Advertising** — predict sales from TV, radio, newspaper budgets (200 markets)

### Time
~75 minutes

## 📐 Part 1 — Simple Linear Regression

**Model:** Y = β₀ + β₁X + ε

- **β₀** (intercept) — predicted Y when X = 0
- **β₁** (slope) — how much Y changes for a one-unit increase in X
- **ε** — random error term, assumed N(0, σ²)

**Goal:** Find β̂₀ and β̂₁ that minimize the Residual Sum of Squares:
```
RSS = Σᵢ (yᵢ - β̂₀ - β̂₁xᵢ)²
```

The OLS (Ordinary Least Squares) solution has a closed form:
```
β̂₁ = Σ(xᵢ - x̄)(yᵢ - ȳ) / Σ(xᵢ - x̄)²
β̂₀ = ȳ - β̂₁x̄
```

In [ ]:
# Simple linear regression: TV → Sales
X_tv = ads[['TV']].values
y = ads['sales'].values

# Fit with sklearn
lr = LinearRegression()
lr.fit(X_tv, y)

print("=== Simple Linear Regression: TV → Sales ===\n")
print(f"β̂₀ (intercept):  {lr.intercept_:.4f}")
print(f"β̂₁ (TV slope):   {lr.coef_[0]:.4f}")
print(f"\nInterpretation:")
print(f"  • Baseline sales (TV=0): {lr.intercept_:.2f} thousand units")
print(f"  • Each $1000 increase in TV spend → +{lr.coef_[0]*1000:.2f} units sold")

# Visualize
tv_range = np.linspace(ads['TV'].min(), ads['TV'].max(), 200).reshape(-1,1)
y_pred_line = lr.predict(tv_range)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Scatter + regression line
axes[0].scatter(ads['TV'], ads['sales'], alpha=0.5, color='#1e5fa8', s=25, edgecolors='white', lw=0.3)
axes[0].plot(tv_range, y_pred_line, color='#e85d20', lw=2.5, label=f'Ŷ = {lr.intercept_:.2f} + {lr.coef_[0]:.4f}·TV')
axes[0].set_xlabel('TV Budget ($000s)')
axes[0].set_ylabel('Sales (000 units)')
axes[0].set_title('Simple Linear Regression: TV → Sales')
axes[0].legend()

# Residuals
y_pred = lr.predict(X_tv)
residuals = y - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.5, color='#6b46c1', s=25, edgecolors='white', lw=0.3)
axes[1].axhline(0, color='#e85d20', lw=1.5, ls='--')
axes[1].set_xlabel('Fitted Values (Ŷ)')
axes[1].set_ylabel('Residuals (y - Ŷ)')
axes[1].set_title('Residual Plot — look for random scatter')

plt.tight_layout()
plt.show()

rse = np.sqrt(np.sum(residuals**2) / (len(y) - 2))
print(f"\nRSE (Residual Standard Error): {rse:.3f} thousand units")
print(f"R²: {r2_score(y, y_pred):.3f} — TV explains {r2_score(y, y_pred)*100:.1f}% of sales variance")

## 🔬 Part 2 — Understanding the Full Regression Output

Using `statsmodels`, we get the complete statistical summary with standard errors, t-statistics, and p-values.

**Key numbers to understand:**
- **coef** — the estimated β
- **std err** — standard error of the estimate (uncertainty)
- **t** — test statistic = coef / std err. Large |t| → evidence the predictor matters
- **P>|t|** — p-value for H₀: βⱼ = 0. Small p → reject null → predictor is significant
- **R²** — proportion of variance in Y explained by the model
- **F-statistic** — tests H₀: all βⱼ = 0 simultaneously. Large F → model explains something

In [ ]:
# Full statistical output with statsmodels
X_with_const = sm.add_constant(ads[['TV']])
model_sm = sm.OLS(ads['sales'], X_with_const).fit()
print(model_sm.summary())

print("\n" + "="*55)
print("READING THE TABLE — line by line:")
print("="*55)
print(f"  const  coef={model_sm.params['const']:.4f} → predicted sales when TV=0")
print(f"  TV     coef={model_sm.params['TV']:.4f}  → +{model_sm.params['TV']*1000:.2f} sales per $1000 TV spend")
print(f"  TV     p={model_sm.pvalues['TV']:.2e}      → extremely significant (p << 0.05)")
print(f"  R²    = {model_sm.rsquared:.3f}          → TV explains {model_sm.rsquared*100:.1f}% of sales variance")
print(f"  F-stat = {model_sm.fvalue:.1f}         → p={model_sm.f_pvalue:.2e} → model is significant")

## 📊 Part 3 — Multiple Linear Regression

With multiple predictors:
```
Y = β₀ + β₁X₁ + β₂X₂ + ... + βₚXₚ + ε
```

Now each βⱼ is the effect of Xⱼ on Y *holding all other predictors constant*. This changes the interpretation: a coefficient in multiple regression is not the same as in simple regression.

In [ ]:
# Multiple regression: all three channels
X_all = ads[['TV','radio','newspaper']]
X_all_const = sm.add_constant(X_all)
model_mult = sm.OLS(ads['sales'], X_all_const).fit()
print(model_mult.summary())

print("\n" + "="*60)
print("COMPARISON: Simple vs Multiple Regression")
print("="*60)

# Simple regressions
for col in ['TV','radio','newspaper']:
    Xs = sm.add_constant(ads[[col]])
    m = sm.OLS(ads['sales'], Xs).fit()
    print(f"  Simple ({col:10s}): β={m.params[col]:.4f}  p={m.pvalues[col]:.4f}  R²={m.rsquared:.3f}")

print(f"\n  Multiple (all):  β_TV={model_mult.params['TV']:.4f}  β_radio={model_mult.params['radio']:.4f}  β_news={model_mult.params['newspaper']:.4f}")
print(f"                   R²={model_mult.rsquared:.3f}  (vs best simple R²={r2_score(ads['sales'], LinearRegression().fit(ads[['TV']], ads['sales']).predict(ads[['TV']])):.3f})")
print("\n📌 Newspaper's coefficient flips sign and becomes insignificant in multiple regression.")
print("   It appears correlated with radio, which actually drives the effect.")

In [ ]:
# Visualize: coefficient comparison and confidence intervals
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Coefficients with confidence intervals
coefs = model_mult.params[1:]  # exclude intercept
ci = model_mult.conf_int().iloc[1:]  # 95% CI

y_pos = range(len(coefs))
axes[0].barh(list(coefs.index), coefs.values,
             xerr=[(coefs - ci[0]).values, (ci[1] - coefs).values],
             color=['#1e5fa8','#e85d20','#888'],
             edgecolor='white', height=0.5, capsize=5)
axes[0].axvline(0, color='black', lw=1, ls='--')
axes[0].set_title('Multiple Regression Coefficients\n(with 95% confidence intervals)')
axes[0].set_xlabel('Coefficient value')

# Actual vs Predicted
y_pred_mult = model_mult.predict(X_all_const)
axes[1].scatter(ads['sales'], y_pred_mult, alpha=0.5, color='#1a7a45', s=25,
                edgecolors='white', lw=0.3)
max_val = max(ads['sales'].max(), y_pred_mult.max())
axes[1].plot([0, max_val], [0, max_val], 'k--', lw=1.5, alpha=0.6, label='Perfect predictions')
axes[1].set_xlabel('Actual Sales')
axes[1].set_ylabel('Predicted Sales')
axes[1].set_title(f'Actual vs Predicted — Multiple Regression (R²={model_mult.rsquared:.3f})')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\n📌 Adding radio and newspaper improves R² from 0.61 to {model_mult.rsquared:.2f}")
print("   Newspaper p-value is 0.86 — no evidence it adds value given TV and radio")

## ✅ Part 4 — Checking Assumptions

Linear regression assumes:
1. **Linearity** — relationship between X and Y is linear
2. **Independence** — observations are independent
3. **Homoscedasticity** — constant variance of residuals
4. **Normality** — residuals are normally distributed

Violations don't always invalidate the model, but they affect inference (p-values, CIs).

In [ ]:
# Four diagnostic plots (equivalent to R's plot.lm)
y_fitted = model_mult.fittedvalues
residuals_mult = model_mult.resid
standardized_res = (residuals_mult - residuals_mult.mean()) / residuals_mult.std()
influence = model_mult.get_influence()
leverage = influence.hat_matrix_diag

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# 1. Residuals vs Fitted
axes[0,0].scatter(y_fitted, residuals_mult, alpha=0.5, color='#1e5fa8', s=20)
axes[0,0].axhline(0, color='#e85d20', lw=1.5, ls='--')
z = np.polyfit(y_fitted, residuals_mult, 2)
p = np.poly1d(z)
x_smooth = np.linspace(y_fitted.min(), y_fitted.max(), 100)
axes[0,0].plot(x_smooth, p(x_smooth), color='#e85d20', lw=1.5, alpha=0.8)
axes[0,0].set_xlabel('Fitted values'); axes[0,0].set_ylabel('Residuals')
axes[0,0].set_title('Residuals vs Fitted\n(should be random — no pattern)')

# 2. Q-Q plot (normality)
stats.probplot(residuals_mult, plot=axes[0,1])
axes[0,1].set_title('Normal Q-Q Plot\n(points should follow diagonal)')
axes[0,1].get_lines()[1].set_color('#e85d20')

# 3. Scale-Location (homoscedasticity)
axes[1,0].scatter(y_fitted, np.sqrt(np.abs(standardized_res)), alpha=0.5, color='#1a7a45', s=20)
axes[1,0].set_xlabel('Fitted values')
axes[1,0].set_ylabel('√|Standardized residuals|')
axes[1,0].set_title('Scale-Location\n(should be flat — equal spread)')

# 4. Leverage
axes[1,1].scatter(leverage, standardized_res, alpha=0.5, color='#6b46c1', s=20)
axes[1,1].axhline(0, color='black', lw=0.8, ls='--')
axes[1,1].axhline(2, color='#e85d20', lw=1, ls=':', alpha=0.7)
axes[1,1].axhline(-2, color='#e85d20', lw=1, ls=':', alpha=0.7)
axes[1,1].set_xlabel('Leverage'); axes[1,1].set_ylabel('Standardized residuals')
axes[1,1].set_title('Residuals vs Leverage\n(outliers + high leverage = influential)')

plt.suptitle('Regression Diagnostic Plots — Multiple Regression', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print("\n📌 Reading the diagnostics:")
print("  Plot 1 (Residuals vs Fitted): slight nonlinearity visible at high fitted values")
print("  Plot 2 (Q-Q): residuals are roughly normal — minor deviation at tails")
print("  Plot 3 (Scale-Location): some heteroscedasticity — variance increases slightly with fitted values")
print("  Plot 4 (Leverage): no highly influential outliers (none past Cook's distance boundary)")

## 🔤 Part 5 — Categorical Predictors

Linear regression needs numeric inputs. Categorical variables are encoded as **dummy variables** (also called one-hot encoding). For a variable with k levels, we create k-1 dummy variables.

In [ ]:
# Create dummy variables for 'Student' status
# pandas get_dummies drops one category automatically (reference category)
credit_dummies = pd.get_dummies(credit[['Balance','Income','Student']], 
                                 drop_first=True, dtype=float)
print("Columns after encoding:", list(credit_dummies.columns))
print("\n'Student_Yes'=1 means the person is a student, 0=not a student\n")

# Fit and interpret
X_credit = sm.add_constant(credit_dummies[['Income','Student_Yes']])
model_credit = sm.OLS(credit_dummies['Balance'], X_credit).fit()
print(model_credit.summary().tables[1])

print("\n📌 Interpretation:")
print(f"  • Each $1000 income increase → balance changes by ${model_credit.params['Income']:.2f}")
print(f"  • Being a student → balance is ${model_credit.params['Student_Yes']:.2f} higher on average")
print(f"    (holding income constant)")

## 💼 Exercise

**Task 1:** Load the Boston Housing dataset (`from sklearn.datasets import fetch_california_housing`). Fit a multiple linear regression predicting median house value from all features. Report R² and identify the 3 most significant predictors.

**Task 2:** Check the regression assumptions for your model from Task 1. Identify one potential violation.

**Task 3:** Add a categorical feature to your regression. The Credit dataset has `Married`, `Student`, `Ethnicity` — pick one, encode it, and interpret the coefficient.

**Task 4:** Split your data 70/30 and compare training R² vs test R². If they differ a lot, why?

In [ ]:
# Exercise workspace
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing(as_frame=True)
df_housing = data.frame
print(df_housing.head())

# YOUR CODE HERE

In [ ]:
# @title 📝 Quiz — Linear Regression
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What does β₁ = 0.05 mean in Y = β₀ + β₁X?
# @markdown **Q2:** What does R² = 0.72 mean in plain English?
# @markdown **Q3:** If the p-value for a coefficient is 0.8, what do you conclude?
# @markdown **Q4:** What does RSE measure?
# @markdown **Q5:** Why do we drop one category when encoding a categorical variable?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit your results